# Examen Práctico - Segundo Parcial

**Imagen trabajada en el notebook:** `histo_1.jpg`

## 0) Cargar una de las imágenes histológicas

In [12]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, morphology, segmentation, measure, util
from scipy.ndimage import binary_fill_holes
import pandas as pd
import math

ruta_imagen = "histo_1.jpg"

imagen_rgb = np.asarray(Image.open(ruta_imagen).convert("RGB")).astype(float) / 255.0

plt.figure(figsize=(6,6))
plt.imshow(imagen_rgb)
plt.title("Imagen original RGB")
plt.axis("off")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'histo_1.jpg'

## 1) Realizar una transformación de color para convertir la imagen al espacio CMYK

In [3]:
imagen_cmyk = np.asarray(Image.open(ruta_imagen).convert("CMYK")).astype(float) / 255.0
canal_magenta = imagen_cmyk[:, :, 1]

plt.figure(figsize=(6,6))
plt.imshow(canal_magenta, cmap="gray")
plt.title("Imagen magenta")
plt.axis("off")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'histo_1.jpg'

## 2) Umbralizar la imagen para separar el fondo y los lúmenes

In [4]:
imagen_suavizada = filters.gaussian(canal_magenta, sigma=1, preserve_range=True)
imagen_uint8 = util.img_as_ubyte(imagen_suavizada)
umbral_otsu = filters.threshold_otsu(imagen_uint8)

# En esta máscara quedan en blanco el fondo y los lúmenes
mascara_con_artefactos = imagen_uint8 < umbral_otsu

plt.figure(figsize=(6,6))
plt.imshow(mascara_con_artefactos, cmap="gray")
plt.title("Máscara con artefactos")
plt.axis("off")
plt.show()

NameError: name 'canal_magenta' is not defined

## 3) Limpiar la imagen eliminando artefactos pequeños

In [5]:
mascara_sin_artefactos = morphology.remove_small_objects(mascara_con_artefactos, min_size=300)

plt.figure(figsize=(6,6))
plt.imshow(mascara_sin_artefactos, cmap="gray")
plt.title("Máscara sin artefactos")
plt.axis("off")
plt.show()

NameError: name 'mascara_con_artefactos' is not defined

## 4) Rellenar con 0s el fondo para quedarnos solo con los lúmenes

In [6]:
alto, ancho = mascara_sin_artefactos.shape
semillas = [(0, 0), (0, ancho-1), (alto-1, 0), (alto-1, ancho-1)]

mascara_fondo = np.zeros_like(mascara_sin_artefactos, dtype=bool)
for semilla in semillas:
    mascara_fondo |= segmentation.flood(mascara_sin_artefactos, semilla, tolerance=0)

mascara_lumenes = mascara_sin_artefactos & ~mascara_fondo

plt.figure(figsize=(6,6))
plt.imshow(mascara_lumenes, cmap="gray")
plt.title("Máscara de lúmenes")
plt.axis("off")
plt.show()

NameError: name 'mascara_sin_artefactos' is not defined

## 5) Rellenar los objetos de los lúmenes

In [7]:
mascara_final = binary_fill_holes(mascara_lumenes)

plt.figure(figsize=(6,6))
plt.imshow(mascara_final, cmap="gray")
plt.title("Máscara final")
plt.axis("off")
plt.show()

NameError: name 'mascara_lumenes' is not defined

## 6) Detectar y dibujar los contornos de los lúmenes sobre la imagen original

In [8]:
imagen_superpuesta = segmentation.mark_boundaries(imagen_rgb, mascara_final, color=(0, 1, 0), mode="thick")

plt.figure(figsize=(6,6))
plt.imshow(imagen_superpuesta)
plt.title("Imagen superpuesta")
plt.axis("off")
plt.show()

NameError: name 'imagen_rgb' is not defined

## 7) Identificar y cropear el lumen más grande

In [9]:
etiquetas = measure.label(mascara_final)
propiedades = sorted(measure.regionprops(etiquetas), key=lambda x: x.area, reverse=True)

lumen_mayor = propiedades[0]
min_fila, min_col, max_fila, max_col = lumen_mayor.bbox
crop_lumen_mayor = imagen_rgb[min_fila:max_fila, min_col:max_col]

plt.figure(figsize=(6,6))
plt.imshow(crop_lumen_mayor)
plt.title("Crop del mayor lumen")
plt.axis("off")
plt.show()

NameError: name 'mascara_final' is not defined

## 8) Extraer 13 características geométricas

In [10]:
area = float(lumen_mayor.area)
area_bounding_box = float((max_fila - min_fila) * (max_col - min_col))
area_convexa = float(getattr(lumen_mayor, "area_convex", lumen_mayor.convex_area))
excentricidad = float(lumen_mayor.eccentricity)
diametro_equivalente = float(getattr(lumen_mayor, "equivalent_diameter_area", lumen_mayor.equivalent_diameter))
extension = float(lumen_mayor.extent)
diametro_feret = float(lumen_mayor.feret_diameter_max)
eje_mayor = float(lumen_mayor.axis_major_length)
eje_menor = float(lumen_mayor.axis_minor_length)
orientacion = float(lumen_mayor.orientation)
perimetro = float(lumen_mayor.perimeter)
solidez = float(lumen_mayor.solidity)
compacidad = float((4 * math.pi * area) / (perimetro ** 2))

tabla = pd.DataFrame({
    "Características": [
        "Área",
        "Área de la bounding box",
        "Área convexa",
        "Excentricidad",
        "Diámetro equivalente",
        "Extensión",
        "Diámetro Feret",
        "Longitud del eje mayor",
        "Longitud del eje menor",
        "Orientación",
        "Perímetro",
        "Solidez",
        "Compacidad"
    ],
    "Valor": [
        round(area, 4),
        round(area_bounding_box, 4),
        round(area_convexa, 4),
        round(excentricidad, 4),
        round(diametro_equivalente, 4),
        round(extension, 4),
        round(diametro_feret, 4),
        round(eje_mayor, 4),
        round(eje_menor, 4),
        round(orientacion, 4),
        round(perimetro, 4),
        round(solidez, 4),
        round(compacidad, 4)
    ]
})

tabla

NameError: name 'lumen_mayor' is not defined